# Experiment 4 (quantum) analysis notebook

Reads saved artifacts from `results/experiment_4/`. The quantum module
itself is not re-run here — use `python -m experiments.exp4_quantum` for
that (it requires the exp2 artifacts).

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
main = pd.read_csv(ROOT / 'results/experiment_4/qae_main.csv')
mc = pd.read_csv(ROOT / 'results/experiment_4/mc_convergence.csv')
sw_m = pd.read_csv(ROOT / 'results/experiment_4/sweep_m.csv')
sw_n = pd.read_csv(ROOT / 'results/experiment_4/sweep_n.csv')
resources = json.loads((ROOT / 'results/experiment_4/resources.json').read_text())
summary = json.loads((ROOT / 'results/experiment_4/summary.json').read_text())
print('contracts:', summary['contracts'])
print('sigma inputs:', {k: round(v,4) for k,v in summary['sigma_inputs'].items()})

## Error decomposition

`calibration` = effect of the input sigma (through analytical BS) vs the
true sigma; `discretization` = grid-induced error at the same sigma;
`qae` = estimation error vs the exact discrete expectation; `total` =
their sum vs the true-sigma reference.

In [ ]:
cols = ['maturity','sigma_input','v_reference','v_model_bs','v_exact_discrete','v_qae',
         'calibration_error','discretization_error','qae_estimation_error','total_error']
main[cols].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
inputs = list(dict.fromkeys(main.sigma_input))
for i, inp in enumerate(inputs):
    sub = main[main.sigma_input==inp]
    cal = sub.calibration_error.abs().mean(); disc = sub.discretization_error.abs().mean(); qae = sub.qae_estimation_error.abs().mean()
    ax.bar(i, cal, label='calibration' if i==0 else None, color='#1f77b4')
    ax.bar(i, disc, bottom=cal, label='discretization' if i==0 else None, color='#ff7f0e')
    ax.bar(i, qae, bottom=cal+disc, label='QAE estimation' if i==0 else None, color='#2ca02c')
ax.set_xticks(range(len(inputs))); ax.set_xticklabels(inputs, rotation=20)
ax.set_ylabel('mean |error|'); ax.legend(); ax.grid(axis='y', alpha=.3)
plt.tight_layout(); plt.show()

## Resolution sweeps (known sigma)

QAE estimation error vs evaluation qubits m (Grover queries =
shots*(2^m-1)) and discretization error vs grid qubits n.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11,4))
ax1.plot(sw_m.m, sw_m.qae_estimation_error.abs(), 'o-', color='#d62728')
ax1.set_xlabel('evaluation qubits m'); ax1.set_ylabel('|QAE estimation error|')
ax1.set_yscale('log'); ax1.grid(alpha=.3)
ax2.plot(sw_n.n, sw_n.discretization_error.abs(), 's-', color='#1f77b4')
ax2.set_xlabel('grid qubits n'); ax2.set_ylabel('|discretization error|')
ax2.grid(alpha=.3); plt.tight_layout(); plt.show()

## MC vs QAE error at matched budgets

Descriptive comparison only — not a speedup or advantage claim
(simulator timings, different oracle models).

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.loglog(mc.n_paths, mc.rmse, 'o-', label='classical MC (RMSE over 20 repeats)')
ax.loglog(sw_m.grover_queries, sw_m.total_error.abs(), 's-', label='QAE total |error|')
ax.set_xlabel('budget (MC paths / Grover queries)'); ax.set_ylabel('|pricing error|')
ax.legend(); ax.grid(True, which='both', alpha=.3)
plt.tight_layout(); plt.show()

## Resource accounting


In [ ]:
print('total circuit qubits:', resources['total_circuit_qubits'])
print('A  depth/size:', resources['state_preparation_a']['depth'], resources['state_preparation_a']['size'])
print('Q  depth/size:', resources['grover_q']['depth'], resources['grover_q']['size'])
print('full depth/size:', resources['full_qae_circuit']['depth'], resources['full_qae_circuit']['size'])
print('Grover queries per shot:', resources['grover_applications_per_shot'])
print('A applications (shots):', resources['total_a_applications'])
print('Aer smoke run:', resources['aer_execution']['status'], f"({resources['aer_execution']['runtime_seconds']:.2f}s)")